In [2]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
import pandas as pd
import sqlite3 # Vamos usar para verificar

# --- INÍCIO DA SOLUÇÃO ---

# 1. Pega o diretório do notebook (ex: '.../projeto-vagas-cd/notebooks')
notebook_dir = os.getcwd()

# 2. Sobe um nível para a RAIZ DO PROJETO (ex: '.../projeto-vagas-cd')
#    ESTA É A LINHA QUE CORRIGE O CAMINHO!
PROJECT_ROOT = os.path.dirname(notebook_dir)

# 3. Carregue o arquivo .env a partir da raiz do projeto
env_path = os.path.join(PROJECT_ROOT, '.env')

if not os.path.exists(env_path):
    raise FileNotFoundError(f"Arquivo .env não encontrado. Eu procurei em: {env_path}")

load_dotenv(dotenv_path=env_path)
print(f"Arquivo .env carregado de: {env_path}")

# 4. Pegue a URL RELATIVA que está no .env
relative_db_url = os.getenv("DATABASE_URL") # 'sqlite:///data/vagas.db'

if not relative_db_url:
    raise ValueError("DATABASE_URL não encontrada no arquivo .env")

# 5. Extraia apenas o caminho do arquivo
relative_path = relative_db_url.split('///')[-1] # 'data/vagas.db'

# 6. Crie o CAMINHO ABSOLUTO E COMPLETO
#    Junta a RAÍZ DO PROJETO com o caminho relativo do banco
#    Ex: 'C:/projeto-vagas-cd' + 'data/vagas.db'
absolute_path = os.path.join(PROJECT_ROOT, relative_path)

# 7. Garante que a pasta 'data' na raiz exista
os.makedirs(os.path.dirname(absolute_path), exist_ok=True)

# 8. Crie a URL FINAL para o SQLAlchemy
absolute_path_str = absolute_path.replace('\\', '/')
FINAL_URL = f"sqlite:///{absolute_path_str}"

# --- FIM DA SOLUÇÃO ---

# --- TESTE FINAL ---
print(f"Conectando ao banco em: {FINAL_URL}")

# Verificando as tabelas ANTES de usar o Polars
try:
    conn_sqlite = sqlite3.connect(absolute_path)
    cursor = conn_sqlite.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    print("Tabelas encontradas no arquivo:", tables)
    conn_sqlite.close()

    if not tables:
        raise ValueError("O arquivo de banco de dados está vazio ou não tem tabelas.")

except Exception as e:
    print(f"Erro ao inspecionar o banco: {e}")


Arquivo .env carregado de: c:\projeto-vagas-cd\.env
Conectando ao banco em: sqlite:///c:/projeto-vagas-cd/data/vagas.db
Tabelas encontradas no arquivo: [('vagas',)]


In [2]:
import polars as pl

df = (
    pl.read_delta("../data/gold_delta/dim_vagas/")
    .unique(subset=["vaga_id"])
    .filter(pl.col("mes_publicacao") == 12)
)

df

vaga_id,titulo,descricao_limpa,empresa,localizacao,modalidade,nivel,area_principal,is_tech,data_expiracao,data_publicacao,url,ano_publicacao,mes_publicacao
str,str,str,str,str,str,i8,str,bool,date,"datetime[μs, UTC]",str,i32,i8
"""10500896""","""Maqueiro(a) Hospitalar""","""Somos a maior cooperativa de s…","""Unimed Grande Florianópolis""","""São José, Santa Catarina""","""presencial""",3,"""seguranca""",false,2026-02-08,2025-12-12 18:14:29.744 UTC,"""https://unimedgrandeflorianopo…",2025,12
"""10489191""","""Agente de Frente e Logística""","""Estamos em busca de um(a) Agen…","""Serhum Consultoria em RH""","""São Luís, Maranhão""","""presencial""",0,"""geral""",false,2026-01-09,2025-12-09 16:14:27.022 UTC,"""https://serhum.gupy.io/job/eyJ…",2025,12
"""10561518""","""Pessoa Estagiaria - Foco Admin…","""Quer ajudar a construir uma so…","""Programa de Estágio da PUCPR""","""Curitiba, Paraná""","""presencial""",4,"""geral""",false,2026-01-26,2025-12-19 15:51:16.217 UTC,"""https://estagiopucpr.gupy.io/j…",2025,12
"""10512536""","""Analista de Eventos | RADISSON…","""• A missão desse profissional …","""Atlantica Hospitality Internat…","""Campinas, São Paulo""","""presencial""",0,"""geral""",false,2026-02-10,2025-12-12 14:18:18.107 UTC,"""https://vagasahi.gupy.io/job/e…",2025,12
"""10500826""","""APRENDIZ DE ATENDENTE DE RESTA…","""#A gente vai amar muito se voc…","""McDonald's Restaurante - Arcos…","""Recife, Pernambuco""","""presencial""",3,"""seguranca""",false,2026-02-08,2025-12-10 21:20:07.550 UTC,"""https://restaurantemc.gupy.io/…",2025,12
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""10479119""","""ATENDENTE DE RESTAURANTE ( AME…","""#A gente vai amar muito se voc…","""McDonald's Restaurante - Arcos…","""Lajeado, Rio Grande do Sul""","""presencial""",3,"""seguranca""",false,2026-02-06,2025-12-08 12:30:28.176 UTC,"""https://restaurantemc.gupy.io/…",2025,12
"""10459414""","""Repórter - RPC Londrina""","""🎤 Você nasceu para contar gran…","""GRPCOM""","""Londrina, Paraná""","""presencial""",0,"""mobile""",false,2026-02-09,2025-12-04 14:08:56.920 UTC,"""https://grpcom.gupy.io/job/eyJ…",2025,12
"""10447361""","""Farmacêutico(a) - Guaratinguet…","""#SejaRDSaúde e vamos promover …","""RD Saúde - Farmácias""","""Guaratinguetá, São Paulo""","""presencial""",2,"""geral""",false,2026-02-01,2025-12-03 00:46:58.126 UTC,"""https://rdsaude-farmacia.gupy.…",2025,12


In [4]:
df.drop(columns=['careerPageUrl','careerPageLogo', ], inplace=True)

In [5]:
# Palavras chave para pequisar na descrição do emprego
keywords = ['Python', 'SQL', 'pipeline', 'ETL', 'data engineering', 'data engineer']
filtered_df = df[df['description'].str.contains('|'.join(keywords), case=False)]

# Palavras que não queremos
exclude_keywords = ["Pl",'estágio', 'intern', 'internship', 'sênior', 'pleno', 'senior', 'sr', 'lead', 'coordenador', 'estagiário', 'Jovem Aprendiz']
filtered_df = filtered_df[~filtered_df['name'].str.contains('|'.join(exclude_keywords), case=False)]

# Se não for remoto, entao deve ser do RJ

In [6]:
desired_first_cols = ['name','careerPageName','jobUrl','location', 'publishedDate', 'description', 'companyName', 'applyUrl']

# 2. Crie uma lista das suas colunas desejadas que REALMENTE existem no DataFrame
existing_first_cols = [col for col in desired_first_cols if col in filtered_df.columns]

# 3. Crie uma lista de todas as OUTRAS colunas
other_cols = [col for col in filtered_df.columns if col not in existing_first_cols]

# 4. Combine as duas listas e reordene o DataFrame
df_rearranged = filtered_df[existing_first_cols + other_cols]

print(df_rearranged)

                                                   name        careerPageName  \
64          Especialista de Desenvolvimento de Produtos      Scanntech Brasil   
115   PESSOA CONSULTORA TECNOLOGIA EDUCACIONAL - CUR...            #VempraFTD   
119                                   Developer Analyst                 Topaz   
121   Analista de Gestão de Identidades e Acessos (I...                 Asaas   
133   Técnico de Suporte Júnior - Atendimento ao Cli...                Benner   
...                                                 ...                   ...   
9654                                Analista de Growth           Confidencial   
9710                            Gerente de Fábrica Ágil               Log Lab   
9729  Promotor técnico Pet Society - Porto Alegre e ...  Vetlog Distribuidora   
9731    Promotor técnico Pet Society - Pelotas e região  Vetlog Distribuidora   
9755                    Supervisor de técnico/comercial  Vetlog Distribuidora   

                           

In [12]:
# Vamos testar a lib rake pra extração de palavras chave
from rake_nltk import Rake
rake = Rake()

def extract_keywords(description):
    rake.extract_keywords_from_text(description)
    return ', '.join(rake.get_ranked_phrases()[:5])  # Retorna as 5 principais frases-chave
df_rearranged['keywords'] = df_rearranged['description'].apply(extract_keywords)
df_rearranged.head()

C:\Users\user\AppData\Local\Temp\ipykernel_69432\88229071.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_rearranged['keywords'] = df_rearranged['description'].apply(extract_keywords)


,name,careerPageName,jobUrl,publishedDate,description,id,companyId,applicationDeadline,isRemoteWork,city,state,country,workplaceType,disabilities,skills,keywords
64,Especialista de Desenvolvimento de Produtos,Scanntech Brasil,https://scanntechbrasil.gupy.io/job/eyJqb2JJZC...,2025-10-31T12:17:58.754Z,Buscamos um(a) Especialista de Produtos de int...,9772243,37162,2025-11-29,0,São Paulo,São Paulo,Brasil,on-site,1,[],especialista de produtos de inteligência de me...
115,PESSOA CONSULTORA TECNOLOGIA EDUCACIONAL - CUR...,#VempraFTD,https://vempraftd.gupy.io/job/eyJqb2JJZCI6OTkw...,2025-10-30T21:48:10.472Z,"Há mais de 120 anos, a FTD Educação tem como m...",9904180,53101,2025-11-30,0,Curitiba,Paraná,Brasil,on-site,1,[],vempraftd e faça parte desse time apaixonado p...
119,Developer Analyst,Topaz,https://cobistopaz.gupy.io/job/eyJqb2JJZCI6OTk...,2025-10-28T15:07:58.958Z,¡En Topaz nos une la tecnología y nos conecta ...,9907461,42212,2025-12-02,0,Bogotá,,Colômbia,hybrid,1,[],informações adicionais ¡ ten en cuesta estos b...
121,Analista de Gestão de Identidades e Acessos (I...,Asaas,https://asaas.gupy.io/job/eyJqb2JJZCI6OTkxMzc0...,2025-10-28T18:02:56.719Z,Se você quer iniciar sua trajetória na área de...,9913747,30728,2025-11-28,1,,,Brasil,remote,0,[],que goste de aprender e tenha vontade de cresc...
133,Técnico de Suporte Júnior - Atendimento ao Cli...,Benner,https://vemserbenner.gupy.io/job/eyJqb2JJZCI6O...,2025-10-28T22:45:38.216Z,Temos a missão de facilitar o dia a dia das pe...,9923774,2358,2025-12-31,0,Blumenau,Santa Catarina,Brasil,on-site,1,[],mail e ferramenta sisconweb e demais ferrament...


In [9]:
import polars as pl
lf = pl.read_parquet("../data/silver/gupy/year=2025/month=11/vagas_20251124.parquet")
lf2 = pl.read_parquet("../data/silver/gupy/year=2025/month=11/vagas_20251125.parquet")
lf2.height + lf.height
# concatenar e remover duplicados baseado em "vaga_id"
lf_final = pl.concat([lf, lf2]).unique(subset=["vaga_id"])
lf_final.height

2920

In [28]:
import polars as pl
lf = pl.scan_delta("../data/silver_delta/gupy").collect()
lf.columns

['vaga_id',
 'company_id',
 'titulo',
 'empresa',
 'empresa_logo',
 'empresa_url',
 'descricao_limpa',
 'localizacao',
 'modalidade',
 'nivel',
 'area_principal',
 'skills_tech',
 'skills_soft',
 'total_skills',
 'is_tech',
 'areas',
 'url',
 'data_publicacao',
 'data_expiracao',
 'aceita_pcd',
 'data_processamento',
 'fonte',
 '_horario_ingestao',
 '_partition_date',
 'ano_publicacao',
 'mes_publicacao']

In [25]:
from pathlib import Path

from deltalake import DeltaTable


delta_path = Path("../data/silver_delta/gupy/")

try:
    # Abre a tabela Delta
    delta_table = DeltaTable(delta_path)

    print(f"Versão atual da tabela Delta: {delta_table.version()}")
    print("\n--- HISTÓRICO DE OPERAÇÕES ---")

    # Mostra o log de todas as operações (WRITE, UPDATE, DELETE, etc.)
    history_df = delta_table.history()
    print(history_df)

except Exception as e:
    print(f"Erro ao ler a tabela Delta: {e}")
    print("Verifique se o caminho está correto e se é uma tabela Delta válida.")

Versão atual da tabela Delta: 22

--- HISTÓRICO DE OPERAÇÕES ---
[{'timestamp': 1764679511891, 'operation': 'WRITE', 'operationParameters': {'partitionBy': '["ano_publicacao","mes_publicacao"]', 'mode': 'Overwrite'}, 'engineInfo': 'delta-rs:py-1.2.1', 'clientVersion': 'delta-rs.py-1.2.1', 'operationMetrics': {'execution_time_ms': 741, 'num_added_files': 2, 'num_added_rows': 12501, 'num_partitions': 0, 'num_removed_files': 1}, 'version': 22}, {'timestamp': 1764677300938, 'operation': 'WRITE', 'operationParameters': {'mode': 'Overwrite', 'partitionBy': '["ano_publicacao","mes_publicacao"]'}, 'engineInfo': 'delta-rs:py-1.2.1', 'clientVersion': 'delta-rs.py-1.2.1', 'operationMetrics': {'execution_time_ms': 48, 'num_added_files': 1, 'num_added_rows': 2073, 'num_partitions': 0, 'num_removed_files': 2}, 'version': 21}, {'timestamp': 1764677299424, 'operation': 'WRITE', 'operationParameters': {'partitionBy': '["ano_publicacao","mes_publicacao"]', 'mode': 'Overwrite'}, 'engineInfo': 'delta-rs:p